# Assignment 11: Production Defense-in-Depth Pipeline
**Student:** Quach Ngoc Quang  
**Course:** AICB-P1 — AI Agent Development  
**Model:** Gemma 4 31B (Native Google API)  
---

## 1. Setup & Environment
We load API keys from `.env` and configure the project path.

In [ ]:
import os
import sys
import asyncio
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# Add src directory to Python path
sys.path.insert(0, str(Path(os.getcwd()).parent / 'src'))

from core.config import setup_api_key
from agents.agent import create_unsafe_agent, create_protected_agent
from guardrails.input_guardrails import InputGuardrailPlugin
from guardrails.output_guardrails import OutputGuardrailPlugin, _init_judge
from guardrails.rate_limiter import RateLimitPlugin
from guardrails.audit_log import AuditLogPlugin
from guardrails.monitoring import MonitoringAlert
from attacks.attacks import adversarial_prompts, run_attacks
from testing.testing import print_comparison

# Initialize API configuration
setup_api_key()
print("Environment initialized. Using Native Gemini API.")

## 2. Unprotected Agent (Baseline)
First, we demonstrate the vulnerability of an agent without guardrails.

In [ ]:
unsafe_agent, unsafe_runner = create_unsafe_agent()
print("Evaluating unsafe agent vulnerability...")
unsafe_results = await run_attacks(unsafe_agent, unsafe_runner)

leaked_count = sum(1 for r in unsafe_results if not r['blocked'])
print(f"\nSummary: Unsafe agent leaked in {leaked_count}/{len(unsafe_results)} attacks.")

## 3. Defense-in-Depth Pipeline Implementation
We implement 4 independent safety layers to protect the agent.

In [ ]:
_init_judge()  # Initialize LLM-as-Judge

plugins = [
    RateLimitPlugin(max_requests=10, window_seconds=60),
    InputGuardrailPlugin(),
    OutputGuardrailPlugin(use_llm_judge=True),
    AuditLogPlugin(log_file="../final_audit_log.json")
]

protected_agent, protected_runner = create_protected_agent(plugins)
print("Production-grade pipeline assembled.")

## 4. Protected Agent Evaluation
Rerunning the same advanced attacks against the protected system.

In [ ]:
print("Evaluating protected agent performance...")
protected_results = await run_attacks(protected_agent, protected_runner)

blocked_count = sum(1 for r in protected_results if r['blocked'])
print(f"\nSummary: Guardrails blocked {blocked_count}/{len(protected_results)} attacks.")

## 5. Security Comparison & Final Report

In [ ]:
print_comparison(unsafe_results, protected_results)

# Export final monitoring status
monitor = MonitoringAlert(plugins=plugins)
monitor.check_metrics()
monitor.print_status()

## 6. Conclusion
The defense-in-depth pipeline successfully mitigated significantly more attacks than the baseline. The multi-layered approach combined with Gemma 4's high-performance safety reasoning provides a robust solution for VinBank's customer service assistant.